# UNet 統一比較：Matrix vs Feature Fusion (含 TIID)

四個模型，統一 loss 函數（ablation-style upstream-only）與評估指標：

| 代號 | 模型 | 輸入 | 事故特徵 | TIID |
|------|------|------|---------|------|
| M0 | UNET-Matrix | VD+Hist | — | — |
| M1 | UNET-DirectConcat | VD+Hist | t15 (44-dim) | ✓ |
| M2 | UNET-MidInject | VD+Hist | t15 (44-dim) | ✓ |
| M3 | UNET-Attention | VD+Hist | t15 (44-dim) | ✓ |

**Loss**：`UpstreamCongestionWeightedLoss`（ablation-style）：  
- upstream free-flow = 2×，upstream congested (<60 km/h) = 5×，downstream = 0（排除）  
- 分母 = 上游有效格子數

**指標**：
- `upstream_MAE` km/h：所有上游有效格子的 MAE
- `congestion_F1`：上游格子的 macro F1，排除無壅塞樣本（報告排除數量）

**切分**：eval_protocol v1（train ≤ 2024-11-30，val 2024-12-01 ~ 2025-02-28）  
**Early stopping**：patience=7，監控 val_loss


In [ ]:
import os, sys, datetime, subprocess
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = '/content/drive/MyDrive/陽交大高速公路計畫/main'
else:
    _here = Path(os.getcwd()).resolve()
    for _p in [_here, *_here.parents]:
        if (_p / 'CLAUDE.md').exists():
            PROJECT_ROOT = str(_p)
            break
    else:
        PROJECT_ROOT = str(_here)

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

RUN_ID = datetime.datetime.now(datetime.UTC).strftime('%Y%m%dT%H%M%SZ')
print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'RUN_ID       = {RUN_ID}')

In [ ]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = Path("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc")
assert font_path.exists(), font_path

fm.fontManager.addfont(str(font_path))
cjk_prop = fm.FontProperties(fname=str(font_path))

def force_cjk_font(fig, font_prop=cjk_prop):
    for text in fig.findobj(match=mpl.text.Text):
        text.set_fontproperties(font_prop)

In [ ]:
subprocess.run(['pip', 'install', '-q', 'pyarrow', 'tqdm'], check=True)
import torch
print('cuda:', torch.cuda.is_available())

In [ ]:
import gc
import json
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}, PyTorch: {torch.__version__}')

## 路徑 & 設定

In [ ]:
DATA_ROOT    = Path(PROJECT_ROOT) / 'data'
OUTPUT_ROOT  = Path(PROJECT_ROOT) / 'output' / 'models' / 'unet_unified_comparison' / RUN_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

NPZ_PATH      = DATA_ROOT / 'accident_samples_5m.npz'
FEATURES_CSV  = Path(PROJECT_ROOT) / 'output' / 'feature_engineering' / 'incident_features_44.csv'
META_PARQUET  = DATA_ROOT / 'accident_samples_5m_metadata.parquet'
HIST_NPZ      = DATA_ROOT / 'historical_matrices_5m.npz'

for p in [NPZ_PATH, FEATURES_CSV, META_PARQUET, HIST_NPZ]:
    assert p.exists(), f'Missing: {p}'
print('All data paths OK')
print(f'Output -> {OUTPUT_ROOT}')

In [ ]:
SPEED_MAX = 140.0
PAD_TARGET = 64

@dataclass
class Config:
    upstream_km:     int   = 25
    downstream_km:   int   = 10
    time_before_min: int   = 30
    time_after_min:  int   = 145
    spatial_res_km:  int   = 1
    time_res_min:    int   = 5
    epochs:          int   = 60        # ceiling; early stopping will stop earlier
    patience:        int   = 7         # early-stopping patience (val_loss)
    batch_size:      int   = 8
    lr:              float = 1e-4
    congestion_kmh:  float = 60.0
    tiid_lambda:     float = 0.3
    train_end_date:  str   = '2024-11-30'
    val_end_date:    str   = '2025-02-28'
    feature_set:     str   = 't15'
    min_filled_ratio: float = 0.8

    @property
    def n_space(self):  return (self.upstream_km + self.downstream_km) // self.spatial_res_km + 1
    @property
    def n_time(self):   return (self.time_before_min + self.time_after_min) // self.time_res_min + 1
    @property
    def n_before(self): return self.time_before_min // self.time_res_min
    @property
    def n_after(self):  return self.n_time - self.n_before
    @property
    def pad_h(self):    return (PAD_TARGET - self.n_space) // 2
    @property
    def pad_w(self):    return (PAD_TARGET - self.n_time)  // 2
    @property
    def upstream_start_idx(self): return self.downstream_km // self.spatial_res_km  # = 10

cfg = Config()
print(f'n_space={cfg.n_space}, n_time={cfg.n_time}, n_before={cfg.n_before}, n_after={cfg.n_after}')
print(f'pad_h={cfg.pad_h}, pad_w={cfg.pad_w}, upstream_start_idx={cfg.upstream_start_idx}')

## 資料載入 & 切分

In [ ]:
from scripts.lib.data_loaders import load_samples
from scripts.lib.windows import SampleWindow, subset

TARGET_WINDOW = SampleWindow(
    upstream_km=cfg.upstream_km, downstream_km=cfg.downstream_km,
    time_before_min=cfg.time_before_min, time_after_min=cfg.time_after_min,
    spatial_resolution_km=cfg.spatial_res_km, resolution_min=cfg.time_res_min,
)

samples = load_samples(data_root=DATA_ROOT, resolution_min=cfg.time_res_min)

# ── Feature CSV ──────────────────────────────────────────────────────────────
feat_df   = pd.read_csv(str(FEATURES_CSV), dtype={'task_id': str})
feat_df['task_id'] = feat_df['task_id'].str.strip()
feat_index = feat_df.set_index('task_id')
feat_cols  = [c for c in feat_df.columns if c != 'task_id']
FEATURE_DIM = len(feat_cols)

_T0_ZERO = frozenset({'injuries','fatalities','lane_blockage_rate',
                       'special_accident','accident_A1','accident_A2','accident_A3'})
_T5_ZERO = frozenset({'injuries','fatalities','lane_blockage_rate'})

def apply_feature_mask(feat_arr, feature_set):
    if feature_set == 't15': return feat_arr
    zero_set = _T0_ZERO if feature_set == 't0' else _T5_ZERO
    avail_zero = {f'avail_{c}' for c in zero_set}
    out = feat_arr.copy()
    for i, col in enumerate(feat_cols):
        if col in zero_set or col in avail_zero:
            out[:, i] = 0.0
    return out

# ── Metadata for time split ───────────────────────────────────────────────────
meta_df = pd.read_parquet(str(META_PARQUET))
meta_df['accident_id'] = meta_df['accident_id'].astype(str)
filled_map = dict(zip(meta_df['accident_id'], meta_df.get('filled_ratio', pd.Series(1.0, index=meta_df.index))))
rtime_map  = dict(zip(meta_df['accident_id'], pd.to_datetime(meta_df['report_time'])))

# ── Subset + align ────────────────────────────────────────────────────────────
matrices, ids, features = [], [], []
for i in range(len(samples)):
    acc_id = str(samples.accident_ids[i])
    if float(filled_map.get(acc_id, 0.0)) < cfg.min_filled_ratio:
        continue
    if acc_id not in feat_index.index:
        continue
    m, _ = subset(samples.matrix[i], samples.window,
                  upstream_km=TARGET_WINDOW.upstream_km, downstream_km=TARGET_WINDOW.downstream_km,
                  time_before_min=TARGET_WINDOW.time_before_min, time_after_min=TARGET_WINDOW.time_after_min)
    matrices.append(m.astype(np.float32))
    ids.append(acc_id)
    features.append(
        pd.to_numeric(feat_index.loc[acc_id, feat_cols], errors='coerce')
          .fillna(0.0).to_numpy(dtype=np.float32)
    )

matrices = np.stack(matrices, axis=0)
ids      = np.asarray(ids)
features = np.stack(features, axis=0)
features = apply_feature_mask(features, cfg.feature_set)
print(f'Aligned samples: {len(ids)}  feature_dim={FEATURE_DIM}')

In [ ]:
# ── Historical baseline matrices ──────────────────────────────────────────────
hist_arr  = np.load(str(HIST_NPZ), allow_pickle=False)
src_win   = SampleWindow(
    upstream_km        = int(hist_arr['window_upstream_km']),
    downstream_km      = int(hist_arr['window_downstream_km']),
    time_before_min    = int(hist_arr['window_time_before_min']),
    time_after_min     = int(hist_arr['window_time_after_min']),
    spatial_resolution_km = int(hist_arr['window_spatial_resolution_km']),
    resolution_min     = int(hist_arr['window_resolution_min']),
)
hist_id_map = {str(a): i for i, a in enumerate(hist_arr['accident_ids'])}
_empty_hist = np.full((cfg.n_space, cfg.n_time), np.nan, dtype=np.float32)

hist_matrices = []
n_missing = 0
for acc_id in ids:
    j = hist_id_map.get(acc_id)
    if j is None:
        hist_matrices.append(_empty_hist.copy()); n_missing += 1; continue
    h, _ = subset(hist_arr['matrix'][j], src_win,
                  upstream_km=TARGET_WINDOW.upstream_km, downstream_km=TARGET_WINDOW.downstream_km,
                  time_before_min=TARGET_WINDOW.time_before_min, time_after_min=TARGET_WINDOW.time_after_min)
    hist_matrices.append(h.astype(np.float32))
hist_matrices = np.stack(hist_matrices, axis=0)
if n_missing: print(f'[warn] {n_missing} accidents missing hist baseline → NaN')
print(f'Hist matrices: {hist_matrices.shape}  coverage={100*(~np.isnan(hist_matrices)).mean():.1f}%')

In [ ]:
# ── eval_protocol v1 time-based split ─────────────────────────────────────────
train_cut = pd.Timestamp(cfg.train_end_date) + pd.Timedelta(days=1)
val_cut   = pd.Timestamp(cfg.val_end_date)   + pd.Timedelta(days=1)
train_idx, val_idx, test_idx = [], [], []
for i, acc_id in enumerate(ids):
    ts = rtime_map.get(acc_id)
    if ts is None or ts < train_cut: train_idx.append(i)
    elif ts < val_cut: val_idx.append(i)
    else: test_idx.append(i)
train_idx = np.array(train_idx); val_idx = np.array(val_idx); test_idx = np.array(test_idx)
print(f'Train: {len(train_idx)}  Val: {len(val_idx)}  Test: {len(test_idx)}')

## Loss, Metrics, Dataset, Models

In [ ]:
# ── Loss: ablation-style upstream-only ────────────────────────────────────────
class UpstreamCongestionWeightedLoss(nn.Module):
    """upstream free-flow=2×, congested=5×, downstream excluded; denom=upstream valid cells."""
    def __init__(self, upstream_start_idx: int, congestion_kmh: float = 60.0):
        super().__init__()
        self.up_start = upstream_start_idx
        self.thresh   = congestion_kmh / SPEED_MAX

    def forward(self, pred, target, valid_mask):
        up_valid = valid_mask[:, self.up_start:, :]
        w = torch.full_like(target[:, self.up_start:, :], 2.0)
        cong = (target[:, self.up_start:, :] < self.thresh).float()
        w = w + 3.0 * cong
        l1 = (pred[:, self.up_start:, :] - target[:, self.up_start:, :]).abs() * w * up_valid
        return l1.sum() / up_valid.sum().clamp(min=1)


criterion = UpstreamCongestionWeightedLoss(cfg.upstream_start_idx, cfg.congestion_kmh)


# ── Metrics ───────────────────────────────────────────────────────────────────
def compute_metrics_batch(pred_np: np.ndarray, tgt_np: np.ndarray, valid_np: np.ndarray) -> list[dict]:
    """Per-sample metrics; pred/tgt/valid: (N, n_space, n_after) normalised [0,1]."""
    up = cfg.upstream_start_idx
    results = []
    for i in range(len(pred_np)):
        v   = valid_np[i].astype(bool)
        p   = pred_np[i];  t = tgt_np[i]
        if not v.any():
            results.append({'mae': np.nan, 'upstream_mae': np.nan,
                             'f1': np.nan, 'has_congestion': False})
            continue
        mae = float(np.abs(p[v] - t[v]).mean() * SPEED_MAX)

        # Upstream zone
        up_v = v[up:, :]; up_p = p[up:, :]; up_t = t[up:, :]
        if not up_v.any():
            results.append({'mae': mae, 'upstream_mae': np.nan,
                             'f1': np.nan, 'has_congestion': False})
            continue
        up_mae = float(np.abs(up_p[up_v] - up_t[up_v]).mean() * SPEED_MAX)

        thr = cfg.congestion_kmh / SPEED_MAX
        pc  = (up_p[up_v] < thr); tc = (up_t[up_v] < thr)
        has_cong = bool(tc.any())
        if has_cong:
            tp = (pc & tc).sum(); fp = (pc & ~tc).sum(); fn = (~pc & tc).sum()
            prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
            f1   = 2 * prec * rec / (prec + rec + 1e-8)
        else:
            f1 = np.nan
        results.append({'mae': mae, 'upstream_mae': up_mae, 'f1': float(f1) if has_cong else np.nan,
                         'has_congestion': has_cong})
    return results


def aggregate_metrics(per_sample: list[dict]) -> dict:
    up_maes = [r['upstream_mae'] for r in per_sample if not np.isnan(r.get('upstream_mae', np.nan))]
    f1s     = [r['f1'] for r in per_sample if r['has_congestion']]
    no_cong = sum(1 for r in per_sample if not r['has_congestion'])
    return {
        'upstream_MAE':        round(float(np.mean(up_maes)), 3) if up_maes else np.nan,
        'congestion_F1':       round(float(np.mean(f1s)),     4) if f1s     else np.nan,
        'no_congestion_count': no_cong,
        'n_samples':           len(per_sample),
    }

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
def _compute_decay_target(speed_norm: np.ndarray) -> np.ndarray:
    """Monotonic non-increasing decay target for TIID head supervision."""
    before = speed_norm[:, :cfg.n_before]; after = speed_norm[:, cfg.n_before:]
    baseline = np.nanmean(before)
    if np.isnan(baseline) or baseline <= 1e-6:
        return np.zeros(after.shape[1], dtype=np.float32)
    # np.nanmean emits "Mean of empty slice" (via warnings.warn, not np errstate)
    # when an entire column is NaN.  Suppress at the Python warnings level.
    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Mean of empty slice", category=RuntimeWarning)
        fut = np.nanmean(after, axis=0)
    d   = np.clip((baseline - fut) / baseline, 0, 1)
    return np.minimum.accumulate(np.nan_to_num(d, 0)).astype(np.float32)


class UnifiedDataset(Dataset):
    def __init__(self, idx):
        self.vd   = matrices[idx]
        self.hist = hist_matrices[idx]
        self.feat = features[idx]
        self.ids  = ids[idx]

    def __len__(self): return len(self.vd)

    def __getitem__(self, i):
        arr  = self.vd[i];  h = self.hist[i]
        snorm = np.where(np.isnan(arr), np.nan, np.clip(arr, 0, SPEED_MAX) / SPEED_MAX)
        hnorm = np.where(np.isnan(h),   np.nan, np.clip(h,   0, SPEED_MAX) / SPEED_MAX)

        vd_spd = np.nan_to_num(snorm.copy(), nan=0.0)
        vd_spd[:, cfg.n_before:] = 0.0
        vd_val = np.zeros((cfg.n_space, cfg.n_time), dtype=np.float32)
        vd_val[:, :cfg.n_before] = (~np.isnan(snorm[:, :cfg.n_before])).astype(np.float32)

        hs = np.nan_to_num(hnorm, nan=0.0)
        hv = (~np.isnan(hnorm)).astype(np.float32)

        x = np.stack([vd_spd, vd_val, hs, hv], axis=0)          # (4, H, W)
        x = torch.from_numpy(x).float()
        x = F.pad(x, (cfg.pad_w, cfg.pad_w, cfg.pad_h, cfg.pad_h), mode='reflect')

        tgt_raw = snorm[:, cfg.n_before:]
        valid   = (~np.isnan(tgt_raw)).astype(np.float32)
        tgt     = np.nan_to_num(tgt_raw, nan=0.0)
        decay   = _compute_decay_target(snorm)

        return (
            x,
            torch.from_numpy(tgt).float(),
            torch.from_numpy(valid).float(),
            torch.from_numpy(self.feat[i]).float(),
            torch.from_numpy(decay).float(),
            str(self.ids[i]),
        )


kw = dict(num_workers=2, pin_memory=(DEVICE == 'cuda'))
train_ds = UnifiedDataset(train_idx)
val_ds   = UnifiedDataset(val_idx)
test_ds  = UnifiedDataset(test_idx)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,  **kw)
val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False, **kw)
test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False, **kw)
print(f'Loaders ready — train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}')

In [ ]:
# ── SimpleUNet (M0: no features, no TIID) ─────────────────────────────────────
class SimpleUNet(nn.Module):
    def __init__(self):
        super().__init__()
        def enc(ic, oc): return nn.Sequential(nn.Conv2d(ic, oc, 4, 2, 1), nn.BatchNorm2d(oc), nn.LeakyReLU(0.2))
        def dec(ic, oc): return nn.Sequential(nn.ConvTranspose2d(ic, oc, 4, 2, 1), nn.BatchNorm2d(oc), nn.ReLU())
        self.enc1 = enc(4,   64);  self.enc2 = enc(64,  128)
        self.enc3 = enc(128, 256); self.enc4 = enc(256, 512)
        self.dec4 = dec(512, 256); self.dec3 = dec(512, 128)
        self.dec2 = dec(256, 64);  self.dec1 = dec(128, 64)
        self.final = nn.Sequential(nn.Conv2d(64, 1, 3, padding=1), nn.Sigmoid())

    def forward(self, x, feat=None, decay_target=None):
        e1=self.enc1(x); e2=self.enc2(e1); e3=self.enc3(e2); e4=self.enc4(e3)
        d4=torch.cat([self.dec4(e4), e3], 1); d3=torch.cat([self.dec3(d4), e2], 1)
        d2=torch.cat([self.dec2(d3), e1], 1); d1=self.dec1(d2)
        out = self.final(d1)
        out = out[:, :, cfg.pad_h:cfg.pad_h+cfg.n_space, cfg.pad_w:cfg.pad_w+cfg.n_time].squeeze(1)
        return out[:, :, cfg.n_before:], None


# ── FeatureFusionUNet (M1–M3: with TIID) ──────────────────────────────────────
class FeatureFusionUNet(nn.Module):
    def __init__(self, method: str, use_tiid: bool = True):
        super().__init__()
        self.method = method
        self.use_tiid = use_tiid
        fd = FEATURE_DIM

        in_ch = 4
        if method == 'direct_concat':
            self.feat2inp = nn.Sequential(nn.Linear(fd, 32), nn.ReLU(), nn.Linear(32, 8), nn.Tanh())
            in_ch = 12
        elif method == 'mid_inject':
            self.feat2bot = nn.Linear(fd, 512)
        elif method == 'attention':
            self.feat2gate = nn.Sequential(nn.Linear(fd, 512), nn.Sigmoid())

        def enc(ic, oc): return nn.Sequential(nn.Conv2d(ic, oc, 4, 2, 1), nn.BatchNorm2d(oc), nn.LeakyReLU(0.2))
        def dec(ic, oc): return nn.Sequential(nn.ConvTranspose2d(ic, oc, 4, 2, 1), nn.BatchNorm2d(oc), nn.ReLU())
        self.enc1=enc(in_ch, 64); self.enc2=enc(64,128)
        self.enc3=enc(128,256);   self.enc4=enc(256,512)
        self.dec4=dec(512,256);   self.dec3=dec(512,128)
        self.dec2=dec(256,64);    self.dec1=dec(128,64)
        self.final=nn.Sequential(nn.Conv2d(64,1,3,padding=1), nn.Sigmoid())

        if use_tiid:
            self.decay_head = nn.Sequential(nn.Linear(512+fd, 256), nn.ReLU(), nn.Linear(256, cfg.n_after))
            self.tiid_scale = nn.Parameter(torch.tensor(0.3))

    def forward(self, x, feat, decay_target=None):
        x_in = x
        if self.method == 'direct_concat':
            fm = self.feat2inp(feat).unsqueeze(-1).unsqueeze(-1).expand(-1,-1,x.shape[2],x.shape[3])
            x_in = torch.cat([x, fm], 1)

        e1=self.enc1(x_in); e2=self.enc2(e1); e3=self.enc3(e2); e4=self.enc4(e3)

        if self.method == 'mid_inject':
            e4 = e4 + self.feat2bot(feat).unsqueeze(-1).unsqueeze(-1)
        elif self.method == 'attention':
            e4 = e4 * self.feat2gate(feat).unsqueeze(-1).unsqueeze(-1)

        d4=torch.cat([self.dec4(e4), e3], 1); d3=torch.cat([self.dec3(d4), e2], 1)
        d2=torch.cat([self.dec2(d3), e1], 1); d1=self.dec1(d2)
        out = self.final(d1)
        out = out[:,:,cfg.pad_h:cfg.pad_h+cfg.n_space, cfg.pad_w:cfg.pad_w+cfg.n_time].squeeze(1)
        base_pred = out[:,:,cfg.n_before:]

        if not self.use_tiid:
            return base_pred, None

        pooled = F.adaptive_avg_pool2d(e4, 1).flatten(1)
        dc = torch.sigmoid(self.decay_head(torch.cat([pooled, feat], 1)))
        dc = torch.cummin(dc, dim=1).values
        ctx    = x[:,0,cfg.pad_h:cfg.pad_h+cfg.n_space, cfg.pad_w:cfg.pad_w+cfg.n_before]
        base   = ctx.mean(dim=(1,2))
        penalty = dc * base.unsqueeze(1) * self.tiid_scale.clamp(min=0)
        pred = (base_pred - penalty.unsqueeze(1).expand(-1, cfg.n_space, -1)).clamp(0, 1)
        return pred, dc


def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
for name, model in [
    ('SimpleUNet',     SimpleUNet()),
    ('DC+TIID', FeatureFusionUNet('direct_concat', True)),
    ('MI+TIID', FeatureFusionUNet('mid_inject',    True)),
    ('ATT+TIID',FeatureFusionUNet('attention',     True)),
]:
    print(f'{name:<20} params={count_params(model):,}')

## Training loop (with early stopping)

In [ ]:
criterion_speed = UpstreamCongestionWeightedLoss(cfg.upstream_start_idx, cfg.congestion_kmh).to(DEVICE)
criterion_decay = nn.L1Loss()


def train_one_model(name: str, model: nn.Module, use_tiid: bool = False) -> dict:
    model = model.to(DEVICE)
    opt   = Adam(model.parameters(), lr=cfg.lr, betas=(0.5, 0.999))
    sched = CosineAnnealingLR(opt, T_max=cfg.epochs, eta_min=1e-6)

    ckpt_dir = OUTPUT_ROOT / name
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    best_loss, patience_ctr = float('inf'), 0
    history = []

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        tr_loss = 0.0
        for x, tgt, msk, feat, decay_t, _ in train_loader:
            x, tgt, msk, feat, decay_t = (
                x.to(DEVICE), tgt.to(DEVICE), msk.to(DEVICE),
                feat.to(DEVICE), decay_t.to(DEVICE)
            )
            opt.zero_grad()
            pred, dc = model(x, feat)
            loss = criterion_speed(pred, tgt, msk)
            if use_tiid and dc is not None:
                loss = loss + cfg.tiid_lambda * criterion_decay(dc, decay_t)
            loss.backward(); opt.step()
            tr_loss += loss.item()
        tr_loss /= max(len(train_loader), 1)
        sched.step()

        model.eval()
        all_pred, all_tgt, all_msk = [], [], []
        vl_loss = 0.0
        with torch.no_grad():
            for x, tgt, msk, feat, decay_t, _ in val_loader:
                x, tgt, msk, feat, decay_t = (
                    x.to(DEVICE), tgt.to(DEVICE), msk.to(DEVICE),
                    feat.to(DEVICE), decay_t.to(DEVICE)
                )
                pred, dc = model(x, feat)
                vl_loss += criterion_speed(pred, tgt, msk).item()
                if use_tiid and dc is not None:
                    vl_loss += cfg.tiid_lambda * criterion_decay(dc, decay_t).item()
                all_pred.append(pred.cpu().numpy())
                all_tgt.append(tgt.cpu().numpy())
                all_msk.append(msk.cpu().numpy())
        vl_loss /= max(len(val_loader), 1)

        # Quick upstream MAE for progress display
        all_p = np.concatenate(all_pred); all_t = np.concatenate(all_tgt); all_m = np.concatenate(all_msk)
        up = cfg.upstream_start_idx; v = all_m[:, up:, :].astype(bool)
        up_mae = float(np.abs(all_p[:, up:, :][v] - all_t[:, up:, :][v]).mean() * SPEED_MAX) if v.any() else float('nan')

        if vl_loss < best_loss:
            best_loss = vl_loss
            patience_ctr = 0
            torch.save({k: v.cpu() for k, v in model.state_dict().items()}, ckpt_dir / 'best.pth')
        else:
            patience_ctr += 1

        history.append({'epoch': epoch, 'train_loss': round(tr_loss, 6),
                         'val_loss': round(vl_loss, 6), 'up_mae': round(up_mae, 3)})
        if epoch % 5 == 0 or epoch == 1:
            print(f'  [{name}] ep{epoch:03d} | tr={tr_loss:.4f} vl={vl_loss:.4f} '
                  f'up_MAE={up_mae:.2f} | patience={patience_ctr}/{cfg.patience}')

        gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()

        if patience_ctr >= cfg.patience:
            print(f'  [{name}] early stop at epoch {epoch}')
            break

    print(f'  [{name}] best_val_loss={best_loss:.6f}')
    return {'name': name, 'best_val_loss': best_loss,
            'history': history, 'ckpt': str(ckpt_dir / 'best.pth')}

## 訓練四個模型

In [ ]:
print('=== M0: UNET-Matrix ===')
model_M0 = SimpleUNet()
result_M0 = train_one_model('UNET-Matrix', model_M0, use_tiid=False)

In [ ]:
print('=== M1: UNET-DirectConcat+TIID ===')
model_M1 = FeatureFusionUNet('direct_concat', use_tiid=True)
result_M1 = train_one_model('UNET-DirectConcat-TIID', model_M1, use_tiid=True)

In [ ]:
print('=== M2: UNET-MidInject+TIID ===')
model_M2 = FeatureFusionUNet('mid_inject', use_tiid=True)
result_M2 = train_one_model('UNET-MidInject-TIID', model_M2, use_tiid=True)

In [ ]:
print('=== M3: UNET-Attention+TIID ===')
model_M3 = FeatureFusionUNet('attention', use_tiid=True)
result_M3 = train_one_model('UNET-Attention-TIID', model_M3, use_tiid=True)

## 評估：載入最佳 checkpoint，收集 per-sample 預測

In [ ]:
def evaluate_model(model: nn.Module, ckpt_path: str, loader: DataLoader) -> list[dict]:
    """Load best checkpoint, run evaluation, return per-sample dicts."""
    model.load_state_dict(torch.load(ckpt_path, map_location='cpu', weights_only=True))
    model = model.to(DEVICE).eval()
    records = []
    with torch.no_grad():
        for x, tgt, msk, feat, _, ids_batch in tqdm(loader, leave=False):
            x, tgt, msk, feat = x.to(DEVICE), tgt.to(DEVICE), msk.to(DEVICE), feat.to(DEVICE)
            pred, dc = model(x, feat)
            # pre-accident observed speed from input channel 0
            before = x[:, 0, cfg.pad_h:cfg.pad_h+cfg.n_space, cfg.pad_w:cfg.pad_w+cfg.n_before].cpu().numpy()
            pred_np = pred.cpu().numpy(); tgt_np = tgt.cpu().numpy(); msk_np = msk.cpu().numpy()
            dc_np   = dc.cpu().numpy() if dc is not None else None   # (B, n_after) or None
            ms = compute_metrics_batch(pred_np, tgt_np, msk_np)
            for i, acc_id in enumerate(ids_batch):
                records.append({
                    'acc_id':  acc_id,
                    'before':  before[i],
                    'pred':    pred_np[i],
                    'target':  tgt_np[i],
                    'valid':   msk_np[i],
                    'dc':      dc_np[i] if dc_np is not None else None,   # TIID decay curve
                    **ms[i],
                })
    return records


MODEL_REGISTRY = [
    ('M0: UNET-Matrix',          model_M0, result_M0),
    ('M1: DirectConcat+TIID',    model_M1, result_M1),
    ('M2: MidInject+TIID',       model_M2, result_M2),
    ('M3: Attention+TIID',       model_M3, result_M3),
]
MODEL_SHORT = ['Matrix', 'DC+TIID', 'MI+TIID', 'ATT+TIID']

val_records = {}   # name -> list of per-sample dicts
for (mname, model, result), short in zip(MODEL_REGISTRY, MODEL_SHORT):
    print(f'Evaluating {mname} ...')
    val_records[short] = evaluate_model(model, result['ckpt'], val_loader)
    agg = aggregate_metrics(val_records[short])
    print(f'  upstream_MAE={agg["upstream_MAE"]:.2f} km/h  '
          f'F1={agg["congestion_F1"]:.4f}  '
          f'(no-cong={agg["no_congestion_count"]}/{agg["n_samples"]})')

## 結果摘要表

In [ ]:
rows = []
for (mname, model, result), short in zip(MODEL_REGISTRY, MODEL_SHORT):
    agg = aggregate_metrics(val_records[short])
    rows.append({
        '模型':            short,
        'best_val_loss':  round(result['best_val_loss'], 6),
        'upstream_MAE':   agg['upstream_MAE'],
        'congestion_F1':  agg['congestion_F1'],
        '無壅塞樣本':      f"{agg['no_congestion_count']}/{agg['n_samples']}",
        '收斂 epoch':      len(result['history']),
    })

summary_df = pd.DataFrame(rows).set_index('模型')
print(summary_df.to_string())

summary_df.reset_index().to_csv(str(OUTPUT_ROOT / 'val_summary.csv'), index=False, encoding='utf-8-sig')
print(f'\n-> {OUTPUT_ROOT / "val_summary.csv"}')

## TIID 衰減曲線分析

TIID 頭輸出 `d(t)` 是一個單調非遞增曲線（0 = 無懲罰，1 = 完全降速）。

- **左圖**：三個 TIID 模型的平均衰減係數 d(t)，±1 std，驗證集全樣本
- **右圖**：折算為速度懲罰 = d(t) × baseline_speed × tiid_scale（km/h）

M0（UNET-Matrix）不含 TIID 頭，不顯示。

In [ ]:
# ── TIID decay curve analysis ─────────────────────────────────────────────────
TIID_SPECS = [
    ('DC+TIID',  model_M1),
    ('MI+TIID',  model_M2),
    ('ATT+TIID', model_M3),
]
TIME_MIN = np.arange(cfg.n_after) * cfg.time_res_min   # 0, 5, ..., 145

fig, (ax_dc, ax_pen) = plt.subplots(1, 2, figsize=(14, 5))

for short, model in TIID_SPECS:
    recs = val_records[short]
    dcs  = np.array([r['dc'] for r in recs if r.get('dc') is not None])  # (N, n_after)
    if len(dcs) == 0:
        print(f'[warn] {short}: no dc data'); continue

    # Retrieve learned tiid_scale from model
    scale = float(model.tiid_scale.clamp(min=0).item())

    # ── Left panel: raw decay fraction d(t) ──────────────────────────────────
    mu_dc = dcs.mean(axis=0)
    sd_dc = dcs.std(axis=0)
    ax_dc.plot(TIME_MIN, mu_dc, label=short, lw=2)
    ax_dc.fill_between(TIME_MIN, (mu_dc - sd_dc).clip(0), (mu_dc + sd_dc).clip(0, 1), alpha=0.15)

    # ── Right panel: speed penalty = d(t) × baseline_speed × scale ───────────
    # baseline_speed: mean of observed pre-accident speed (channel 0 of input)
    baselines = np.array([
        r['before'].mean() * SPEED_MAX   # before is normalised [0,1]
        for r in recs if r.get('dc') is not None
    ])  # (N,) in km/h
    pen = dcs * baselines[:, None] * scale   # (N, n_after) in km/h
    mu_pen = pen.mean(axis=0)
    sd_pen = pen.std(axis=0)
    ax_pen.plot(TIME_MIN, mu_pen, label=f'{short}  (scale={scale:.3f})', lw=2)
    ax_pen.fill_between(TIME_MIN, (mu_pen - sd_pen).clip(0), mu_pen + sd_pen, alpha=0.15)

# ── Formatting ────────────────────────────────────────────────────────────────
ax_dc.set_xlim(0, TIME_MIN[-1])
ax_dc.set_ylim(0, 1)
ax_dc.axhline(0, color='gray', lw=0.5)
ax_dc.set_xlabel('事故後時間（分鐘）', fontsize=10)
ax_dc.set_ylabel('TIID 衰減係數 d(t)', fontsize=10)
ax_dc.set_title('平均 TIID 衰減曲線（驗證集全樣本，±1 std）', fontsize=11)
ax_dc.legend(fontsize=9); ax_dc.grid(alpha=0.3)

ax_pen.set_xlim(0, TIME_MIN[-1])
ax_pen.set_ylim(bottom=0)
ax_pen.set_xlabel('事故後時間（分鐘）', fontsize=10)
ax_pen.set_ylabel('速度懲罰（km/h）', fontsize=10)
ax_pen.set_title('TIID 速度懲罰 = d(t) × 事前基線速 × scale', fontsize=11)
ax_pen.legend(fontsize=9); ax_pen.grid(alpha=0.3)

fig.suptitle('TIID 衰減頭輸出分析（驗證集）', fontsize=13)
plt.tight_layout()
force_cjk_font(fig)
force_cjk_font(ax_dc)
force_cjk_font(ax_pen)
plt.savefig(str(OUTPUT_ROOT / 'tiid_curves.png'), dpi=130, bbox_inches='tight')
plt.show()
print('Saved tiid_curves.png')

# ── Print learned tiid_scale per model ───────────────────────────────────────
print('\n── 學習到的 tiid_scale ──')
for short, model in TIID_SPECS:
    print(f'  {short:<12} tiid_scale = {model.tiid_scale.item():.4f}')

In [ ]:
# ── Loss curves for all 4 models ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharey=False)
for ax, (mname, model, result), short in zip(axes, MODEL_REGISTRY, MODEL_SHORT):
    h = result['history']
    epochs = [r['epoch'] for r in h]
    tr = [r['train_loss'] for r in h]
    vl = [r['val_loss']   for r in h]
    best = result['best_val_loss']
    ax.plot(epochs, tr, label='train', lw=1.5)
    ax.plot(epochs, vl, label='val',   lw=1.5)
    ax.axhline(best, color='gray', ls='--', lw=0.8, alpha=0.7)
    ax.set_title(f'{short}\nbest={best:.4f}', fontsize=9)
    ax.set_xlabel('Epoch', fontsize=8); ax.set_ylabel('Loss', fontsize=8)
    ax.tick_params(labelsize=7); ax.legend(fontsize=7); ax.grid(alpha=0.3)
fig.suptitle('訓練 / 驗證 Loss 曲線', fontsize=12)
plt.tight_layout()
force_cjk_font(fig)
plt.savefig(str(OUTPUT_ROOT / 'loss_curves.png'), dpi=130, bbox_inches='tight')
plt.show()
print('Saved loss_curves.png')

## Best / Worst 3 Cases per Model — Side-by-Side Comparison

每個模型依 `upstream_MAE` 選出最好 3 個（誤差最小）和最差 3 個（誤差最大），  
每個 case 同時顯示所有四個模型的預測，便於橫向比較。

圖格排列（每行為一個 case）：  
`事故前速度場` | `實際（後）` | `Matrix` | `DC+TIID` | `MI+TIID` | `ATT+TIID`

In [ ]:
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

# Custom colormap: blue=congested (<60), green/yellow=moderate, white=free-flow
_SPEED_CMAP = plt.cm.RdYlGn

# Spatial & time tick positions
SPACE_KM = np.linspace(
    -cfg.downstream_km, cfg.upstream_km, cfg.n_space
)  # -10 to +25 km
TIME_POST_MIN = np.arange(cfg.n_after) * cfg.time_res_min   # 0, 5, ..., 145
TIME_PRE_MIN  = (np.arange(cfg.n_before) - cfg.n_before) * cfg.time_res_min  # -30,...,-5


def _heatmap(ax, data_norm, title, is_error=False, vmin=None, vmax=None):
    """Render a normalised speed matrix as a colour heatmap."""
    if is_error:
        d = data_norm * SPEED_MAX
        vm = 40.0 if vmax is None else vmax
        im = ax.imshow(d, aspect='auto', cmap='RdBu_r',
                       vmin=-vm, vmax=vm, origin='lower', interpolation='nearest')
    else:
        d = np.clip(data_norm, 0, 1) * SPEED_MAX
        im = ax.imshow(d, aspect='auto', cmap=_SPEED_CMAP,
                       vmin=0, vmax=SPEED_MAX, origin='lower', interpolation='nearest')
    ax.set_title(title, fontsize=7, pad=2)
    ax.tick_params(labelsize=5, length=2)
    # Draw accident-point line on spatial axis
    ax.axhline(cfg.upstream_start_idx - 0.5, color='black', lw=0.8, ls='--', alpha=0.5)
    return im


def _add_axis_labels(ax, which):
    """Add km / min tick labels on selected axes."""
    n_rows = len(SPACE_KM)
    if which in ('left', 'all'):
        yticks = np.linspace(0, n_rows-1, 5, dtype=int)
        ax.set_yticks(yticks)
        ax.set_yticklabels([f'{SPACE_KM[t]:.0f}' for t in yticks], fontsize=5)
        ax.set_ylabel('km', fontsize=6)
    else:
        ax.set_yticks([])
    if which in ('bottom', 'all'):
        ax.set_xlabel('分鐘', fontsize=6)


print('Heatmap utilities ready.')

In [ ]:
# Build per-accident lookup from each model (acc_id → record)
lookup = {short: {r['acc_id']: r for r in recs}
          for short, recs in val_records.items()}

N_SHOW = 3  # best N + worst N per model

COLUMN_LABELS = ['事故前', '實際（後）'] + MODEL_SHORT
COL_DESCRIPTIONS = ['事故前\n觀測速度', '事故後\n實際速度',
                     'Matrix\n預測', 'DC+TIID\n預測', 'MI+TIID\n預測', 'ATT+TIID\n預測']

def make_comparison_figure(focus_model_short: str, title: str) -> plt.Figure:
    """6 rows × 6 columns figure: best-N-worst-N cases of focus_model, all 4 models shown."""
    recs = val_records[focus_model_short]
    # Sort by upstream_mae (NaN last)
    valid_recs = [r for r in recs if not np.isnan(r.get('upstream_mae', np.nan))]
    if not valid_recs:
        print(f'[warn] {focus_model_short}: no valid upstream_mae values'); return None
    sorted_recs = sorted(valid_recs, key=lambda r: r['upstream_mae'])
    n_avail = len(sorted_recs)
    best_k  = sorted_recs[:min(N_SHOW, n_avail)]
    worst_k = sorted_recs[max(0, n_avail - N_SHOW):]
    cases   = best_k + worst_k
    n_cases = len(cases)

    n_cols = 6  # pre | truth | M0 | M1 | M2 | M3
    fig, axes = plt.subplots(
        n_cases + 1, n_cols,  # +1 for header row
        figsize=(n_cols * 3.2, n_cases * 2.8 + 0.8),
        gridspec_kw={'height_ratios': [0.15] + [1.0] * n_cases}
    )
    if n_cases == 1:
        axes = axes[np.newaxis, :]

    # Header row: column labels
    for c, lbl in enumerate(COL_DESCRIPTIONS):
        axes[0, c].axis('off')
        axes[0, c].text(0.5, 0.5, lbl, ha='center', va='center',
                        fontsize=8, fontweight='bold', transform=axes[0, c].transAxes)

    for row_i, case_rec in enumerate(cases):
        acc_id   = case_rec['acc_id']
        kind     = '最佳' if row_i < len(best_k) else '最差'
        up_mae   = case_rec['upstream_mae']
        f1_val   = case_rec.get('f1', np.nan)
        f1_str   = f'{f1_val:.3f}' if not np.isnan(f1_val) else 'n/a'
        row_ax   = axes[row_i + 1]

        # Col 0: pre-accident speed
        pre_data = case_rec['before']  # (n_space, n_before) normalised
        _heatmap(row_ax[0], pre_data, f'{kind} | {acc_id[:8]}…\nupMAE={up_mae:.1f} F1={f1_str}')
        _add_axis_labels(row_ax[0], 'left')

        # Col 1: ground truth post-accident
        truth = case_rec['target']  # (n_space, n_after) normalised
        _heatmap(row_ax[1], truth, '實際')
        _add_axis_labels(row_ax[1], 'none')

        # Cols 2–5: each model's prediction
        for col_i, short in enumerate(MODEL_SHORT):
            m_rec  = lookup[short].get(acc_id)
            if m_rec is None:
                row_ax[col_i + 2].axis('off')
                continue
            m_up_mae = m_rec.get('upstream_mae', np.nan)
            label    = f'{short}\n{m_up_mae:.1f} km/h' if not np.isnan(m_up_mae) else short
            _heatmap(row_ax[col_i + 2], m_rec['pred'], label)
            _add_axis_labels(row_ax[col_i + 2], 'none')

    # Divider line between best and worst
    if len(best_k) < n_cases:
        split_row = len(best_k) + 1
        fig.add_artist(
            plt.Line2D(
                [0.01, 0.99],
                [1 - split_row / (n_cases + 1), 1 - split_row / (n_cases + 1)],
                transform=fig.transFigure, color='navy', lw=1.5, ls='--', alpha=0.5
            )
        )

    fig.suptitle(title, fontsize=11, y=1.005)
    plt.tight_layout(rect=[0, 0, 1, 1])
    return fig


print(f'Case selection ready. Will show best {N_SHOW} + worst {N_SHOW} per model.')

In [ ]:
fig0 = make_comparison_figure(
    'Matrix',
    f'M0 UNET-Matrix 觀點 — 最佳 {N_SHOW} / 最差 {N_SHOW} (按 upstream MAE)'
)
force_cjk_font(fig0)
if fig0:
    fig0.savefig(str(OUTPUT_ROOT / 'cases_M0_Matrix.png'), dpi=130, bbox_inches='tight')
    plt.show()
print('Saved cases_M0_Matrix.png')

In [ ]:
fig1 = make_comparison_figure(
    'DC+TIID',
    f'M1 DirectConcat+TIID 觀點 — 最佳 {N_SHOW} / 最差 {N_SHOW}'
)
force_cjk_font(fig1)
if fig1:
    fig1.savefig(str(OUTPUT_ROOT / 'cases_M1_DC_TIID.png'), dpi=130, bbox_inches='tight')
    plt.show()
print('Saved cases_M1_DC_TIID.png')

In [ ]:
fig2 = make_comparison_figure(
    'MI+TIID',
    f'M2 MidInject+TIID 觀點 — 最佳 {N_SHOW} / 最差 {N_SHOW}'
)
force_cjk_font(fig2)
if fig2:
    fig2.savefig(str(OUTPUT_ROOT / 'cases_M2_MI_TIID.png'), dpi=130, bbox_inches='tight')
    plt.show()
print('Saved cases_M2_MI_TIID.png')

In [ ]:
fig3 = make_comparison_figure(
    'ATT+TIID',
    f'M3 Attention+TIID 觀點 — 最佳 {N_SHOW} / 最差 {N_SHOW}'
)
force_cjk_font(fig3)
if fig3:
    fig3.savefig(str(OUTPUT_ROOT / 'cases_M3_ATT_TIID.png'), dpi=130, bbox_inches='tight')
    plt.show()
print('Saved cases_M3_ATT_TIID.png')

## 儲存完整結果 JSON

In [ ]:
def _serialisable(r: dict) -> dict:
    """Strip large array fields from a per-sample record before JSON serialisation."""
    _SKIP = {'before', 'pred', 'target', 'valid', 'dc'}
    return {k: v for k, v in r.items() if k not in _SKIP}

all_results = {}
for (mname, model, result), short in zip(MODEL_REGISTRY, MODEL_SHORT):
    agg = aggregate_metrics(val_records[short])
    all_results[short] = {
        'model_name':    mname,
        'best_val_loss': result['best_val_loss'],
        'converged_ep':  len(result['history']),
        'val_aggregate': agg,
        'val_per_sample': [_serialisable(r) for r in val_records[short]],
        'history': result['history'],
    }

out_json = OUTPUT_ROOT / 'all_results.json'
with open(str(out_json), 'w', encoding='utf-8') as f:
    json.dump({'run_id': RUN_ID, 'results': all_results}, f, indent=2,
              ensure_ascii=False, default=lambda x: float(x) if hasattr(x,'item') else str(x))
print(f'Full results -> {out_json}')

print('\n=== Final Val Summary ===')
print(summary_df.to_string())